## Feature Engineering

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/tracking_ice_processed.csv')

### Pivoting to use a different Target Variable
The first attempt at training the model produced a very very low Accuracy score. The low number of observations and the high cardinality of the Target Class proved to be too much to overcome. Taking a step back, I want to see if we can still utilize the same dataset but this time choosing a Target Variable with less Cardinality.

In [2]:
# Checking the overall DataFrame
df.info()

# Printing the first 5 rows to get an idea of what the data looks like
display(df.head(5))


<class 'pandas.DataFrame'>
RangeIndex: 169 entries, 0 to 168
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   month           169 non-null    str  
 1   event           169 non-null    str  
 2   day_with_month  169 non-null    str  
 3   city            169 non-null    str  
 4   zip_code        169 non-null    int64
 5   year            169 non-null    int64
dtypes: int64(2), str(4)
memory usage: 8.1 KB


,month,event,day_with_month,city,zip_code,year
0,January,"On January 7, the day that the United States C...",January 7,"Kern County, California",93387,2025
1,January,"On January 23, high-profile US Immigration and...",January 23,"Atlanta, Georgia",30301,2025
2,January,"On January 23, high-profile US Immigration and...",January 23,"Boston, Massachusetts",2101,2025
3,January,"On January 23, high-profile US Immigration and...",January 23,"Denver, Colorado",80201,2025
4,January,"On January 23, high-profile US Immigration and...",January 23,"Miami, Florida",33101,2025


Performing a lot of the same Feature Engineering with the previous attempt in-terms of datetime column casting

In [3]:
# Combining the year and month day into a single column, to be able to cast as a datetime object
df['month_day_year'] = df['day_with_month'].str.cat(df['year'].astype(str), sep=' ')

# Dropping the day_with_month and year columns now, since we have a combined column to capture the data
df.drop(columns=['day_with_month', 'year'], inplace=True)

# Converting the month_day_year column from string to datetime
df["date"] = pd.to_datetime(df["month_day_year"], format="%B %d %Y")

# Dropping the month_day_year now after creating the new column
df.drop(columns=["month_day_year"], inplace=True)

# Ensure that the DataFrame is sorted by the date column
df.sort_values(by='date', inplace=True)


Now, here is where we pivot. Instead of continuing to use the `zip_code` column, we will extract the State from the `city` column. The values are a string in the format: `{city}, {state}`

In [4]:
# Since we're working with strings, we want to make sure we strip any leading and trailing whitespace
df['city'] = df['city'].str.strip()

# Extracting the State and saving into a new column
df['state'] = df['city'].str.extract(r',\s*(.*)')

# Validating the new column
df.info()
display(df[df.isnull().any(axis=1)])

# Found there is a Null value in the States! Updating that one specific cell so it is no longer Null
df.loc[148, 'state'] = 'Maryland'

# Checking again for nulls
display(df.info())
display(df.head(5))
display(df[df.isnull().any(axis=1)])

# Nulls gone! Now we can proceed to cleaning up our Dataframe
# Cleaning the whitespace in both month and state just in case
df['state'] = df['state'].str.strip()
df['month'] = df['month'].str.strip()


<class 'pandas.DataFrame'>
Index: 169 entries, 0 to 168
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   month     169 non-null    str           
 1   event     169 non-null    str           
 2   city      169 non-null    str           
 3   zip_code  169 non-null    int64         
 4   date      169 non-null    datetime64[us]
 5   state     168 non-null    str           
dtypes: datetime64[us](1), int64(1), str(4)
memory usage: 9.2 KB


,month,event,city,zip_code,date,state
148,January,"On January 3, ICE arrested a mother in Marylan...",Maryland,20601,2026-01-03,NaN


<class 'pandas.DataFrame'>
Index: 169 entries, 0 to 168
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   month     169 non-null    str           
 1   event     169 non-null    str           
 2   city      169 non-null    str           
 3   zip_code  169 non-null    int64         
 4   date      169 non-null    datetime64[us]
 5   state     169 non-null    str           
dtypes: datetime64[us](1), int64(1), str(4)
memory usage: 13.3 KB


None

,month,event,city,zip_code,date,state
0,January,"On January 7, the day that the United States C...","Kern County, California",93387,2025-01-07,California
8,January,"On January 23, high-profile US Immigration and...","Seattle, Washington",98101,2025-01-23,Washington
7,January,"On January 23, high-profile US Immigration and...","Philadelphia, Pennsylvania",19019,2025-01-23,Pennsylvania
6,January,"On January 23, high-profile US Immigration and...","Newark, New Jersey",7101,2025-01-23,New Jersey
5,January,"On January 23, high-profile US Immigration and...","New York City, New York",10001,2025-01-23,New York


,month,event,city,zip_code,date,state


In [5]:
# Dropping the Zip Code column and the City column now, since we are focusing on just the State. We do NOT want the model inadvertintely treating the zip_code as ordinal data
# NOTE: Making this its own cell because re-running the previous code will end up breaking lol
df.drop(columns=['city', 'zip_code'], inplace=True)

In [6]:
# We do the same as we did previously, but instead of zip_code we do it for the state. In order to create a chronological order of the States.
df['previous_state'] = df['state'].shift(1)   # There's already a built-in function that does this!!!!!!

In [7]:
# Dropping the VERY first row, because it contains the previous_state as a null value
# This null value could introduce errors or bad training if incorporated into the training dataset, so it's safest to simply drop it
df.dropna(inplace=True)

In [8]:
df.head()

,month,event,date,state,previous_state
8,January,"On January 23, high-profile US Immigration and...",2025-01-23,Washington,California
7,January,"On January 23, high-profile US Immigration and...",2025-01-23,Pennsylvania,Washington
6,January,"On January 23, high-profile US Immigration and...",2025-01-23,New Jersey,Pennsylvania
5,January,"On January 23, high-profile US Immigration and...",2025-01-23,New York,New Jersey
9,January,"On January 23, high-profile US Immigration and...",2025-01-23,D.C.,New York


## Model Training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# One Hot Encoding our previous_state column. This is because we want to make sure the model treats these Features as categorical, and not ordinal
dummy_variables = pd.get_dummies(df['previous_state'], prefix="is", drop_first=False).astype(int)

# Now, combining the One Hot Encoded columns back to the original dataframe, but dropping the previous_state column
# This helps by converting our Features into 0's or 1's, which our model understands better
df_encoded = pd.concat([df.drop(columns=['previous_state']), dummy_variables], axis=1)

display(df_encoded.columns)

# One Hot Encoding our month column. This is because we want to make sure the model treats these Features as categorical, and not ordinal
dummy_months = pd.get_dummies(df_encoded['month'], prefix="is", drop_first=False).astype(int)

# Now, combining the One Hot Encoded columns back to the original dataframe, but dropping the previous_state column
# This helps by converting our Features into 0's or 1's, which our model understands better
df_encoded = pd.concat([df_encoded.drop(columns=['month']), dummy_months], axis=1)

display(df_encoded.columns)


# Splitting our data into X and y, where X is all of our Features and y is our target variable
# Dropping state because it is the Target Variable
# Dropping event because it is simply a long string description.
# Dropping date becuase we have built the chronological order with sorting + previous_state
X = df_encoded.drop(columns=["state", "event", "date"])
y = df_encoded["state"]

# Creating our test/train data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=23,
    shuffle=False   # Shuffle FALSE becase we want to keep the chronological order
)

# Instantiating our RandomForestClassifier model
rf_model = RandomForestClassifier()

# Training our model (we DON'T Transform on Categorical values, only Fit!)
rf_model.fit(X_train, y_train)

# Making predictions
y_pred = rf_model.predict(X_test)

Index(['month', 'event', 'date', 'state', 'is_Alabama', 'is_Arizona',
       'is_California', 'is_Colorado', 'is_Connecticut', 'is_D.C.',
       'is_Florida', 'is_Georgia', 'is_Idaho', 'is_Illinois', 'is_Iowa',
       'is_Kansas', 'is_Louisiana', 'is_Maine', 'is_Maryland',
       'is_Massachusetts', 'is_Michigan', 'is_Minnesota', 'is_New Jersey',
       'is_New York', 'is_North Carolina', 'is_Oklahoma', 'is_Oregon',
       'is_Pennsylvania', 'is_Tennessee', 'is_Texas', 'is_Utah', 'is_Vermont',
       'is_Washington'],
      dtype='str')

Index(['event', 'date', 'state', 'is_Alabama', 'is_Arizona', 'is_California',
       'is_Colorado', 'is_Connecticut', 'is_D.C.', 'is_Florida', 'is_Georgia',
       'is_Idaho', 'is_Illinois', 'is_Iowa', 'is_Kansas', 'is_Louisiana',
       'is_Maine', 'is_Maryland', 'is_Massachusetts', 'is_Michigan',
       'is_Minnesota', 'is_New Jersey', 'is_New York', 'is_North Carolina',
       'is_Oklahoma', 'is_Oregon', 'is_Pennsylvania', 'is_Tennessee',
       'is_Texas', 'is_Utah', 'is_Vermont', 'is_Washington', 'is_April',
       'is_August', 'is_December', 'is_February', 'is_January', 'is_July',
       'is_June', 'is_March', 'is_May', 'is_November', 'is_October',
       'is_September'],
      dtype='str')

8%!!!! That's an improvement over our previous 5%

In [10]:
# Checking the Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

Accuracy: 0.08823529411764706


Taking a look and comparing the actual test data set vs the predictions.

In [17]:
display(df_encoded.info())

display(pd.DataFrame({"Actual": y_test.values, "Predicted": y_pred}))

<class 'pandas.DataFrame'>
Index: 168 entries, 8 to 168
Data columns (total 44 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   event              168 non-null    str           
 1   date               168 non-null    datetime64[us]
 2   state              168 non-null    str           
 3   is_Alabama         168 non-null    int64         
 4   is_Arizona         168 non-null    int64         
 5   is_California      168 non-null    int64         
 6   is_Colorado        168 non-null    int64         
 7   is_Connecticut     168 non-null    int64         
 8   is_D.C.            168 non-null    int64         
 9   is_Florida         168 non-null    int64         
 10  is_Georgia         168 non-null    int64         
 11  is_Idaho           168 non-null    int64         
 12  is_Illinois        168 non-null    int64         
 13  is_Iowa            168 non-null    int64         
 14  is_Kansas          168 non

None

,Actual,Predicted
0,Minnesota,Louisiana
1,Michigan,Louisiana
2,New York,New York
3,Maryland,Louisiana
4,Illinois,Louisiana
5,Texas,Louisiana
6,Minnesota,Louisiana
7,Minnesota,Louisiana
8,Minnesota,Louisiana
9,Maryland,Louisiana


### Saving the model

In [16]:
import joblib

joblib.dump(rf_model, "../model/icepredictor.joblib")

['../model/icepredictor.joblib']